# LLM APIs, Reasoning Flows & Prompt-to-Action Pipelines

**Track:** Applied Agent Engineering Foundations  
**Module:** LLM Foundations  
**Environment:** SageMaker Jupyter Notebook  
**Audience:** Software engineers building enterprise AI workflows on AWS

## What learners will learn
By the end of this lab, learners will be able to:
1. Connect securely from SageMaker to Amazon Bedrock.
2. Discover available Bedrock foundation models from code.
3. Use Bedrock LLM APIs for prompt experiments and structured generation.
4. Build a prompt-to-action pipeline with tool/function-calling style behavior.
5. Validate model output before execution.
6. Manage context windows using chunking and budget checks.
7. Add safe error handling and run logging for enterprise usage.

## Instructor flow
- Part A: Secure setup and model discovery
- Part B: Prompt styles and reasoning flows
- Part C: Prompt-to-action pipeline and tool/function calling pattern
- Part D: Context window management
- Part E: Error handling, validation, and mini-hack

## Before you run

This notebook assumes:
- you are running inside **SageMaker Jupyter Notebook**
- your notebook has an **execution role** with Bedrock and S3 read permissions
- Bedrock model access is already enabled for the models you want to test
- your learning files are already uploaded to **S3**
- you are **reading** from S3 only in this notebook; no S3 write-back is required

### Design choice for this lab
This notebook teaches a **model-agnostic prompt-to-action pipeline**.  
That means learners first understand:
- how to call an LLM API
- how to get structured output
- how to validate it
- how to safely call backend functions

Later, the same pattern can be extended to native tool use, agents, workflows, or orchestration frameworks.

In [42]:
# Uncomment only if your environment is missing any package
# %pip install -q boto3 pandas

import json
import re
import textwrap
from typing import Any, Dict, List, Optional

import boto3
import pandas as pd
from botocore.exceptions import ClientError


# Get AWS_REGION from the current boto3 session
AWS_REGION = boto3.Session().region_name 

# Bedrock clients
bedrock = boto3.client("bedrock", region_name=AWS_REGION)
bedrock_runtime = boto3.client("bedrock-runtime", region_name=AWS_REGION)

# Other AWS clients we will use in this notebook
s3 = boto3.client("s3", region_name=AWS_REGION)
sts = boto3.client("sts", region_name=AWS_REGION)


# Print out the configuration to verify everything is set up correctly
print("AWS Region:", AWS_REGION)

AWS Region: us-east-1


## Step 1 — Show available models in Bedrock

**Goal:** Let learners see what Bedrock exposes in the current region.

**Discuss:**
- model access can vary by account and region
- some models support chat, some embeddings, some image or multimodal inputs
- classroom code should **discover** available models instead of relying on memory

In [43]:
def list_bedrock_models() -> pd.DataFrame:
    response = bedrock.list_foundation_models()
    rows = []

    for item in response.get("modelSummaries", []):
        rows.append({
            "provider": item.get("providerName"),
            "model_id": item.get("modelId"),
            "model_name": item.get("modelName"),
            "input_modalities": ", ".join(item.get("inputModalities", [])),
            "output_modalities": ", ".join(item.get("outputModalities", [])),
            "streaming": item.get("responseStreamingSupported"),
            "customizations": ", ".join(item.get("customizationsSupported", [])),
            "inference_types": ", ".join(item.get("inferenceTypesSupported", [])),
        })

    df = pd.DataFrame(rows).sort_values(["provider", "model_id"]).reset_index(drop=True)
    return df

try:
    models_df = list_bedrock_models()
    with pd.option_context('display.max_rows', 250):
        display(models_df)
except ClientError as e:
    print("Could not list models. Check Bedrock permissions and regional access.")
    print(e)

,provider,model_id,model_name,input_modalities,output_modalities,streaming,customizations,inference_types
0,AI21 Labs,ai21.jamba-1-5-large-v1:0,Jamba 1.5 Large,TEXT,TEXT,True,,ON_DEMAND
1,AI21 Labs,ai21.jamba-1-5-mini-v1:0,Jamba 1.5 Mini,TEXT,TEXT,True,,ON_DEMAND
2,Amazon,amazon.nova-2-lite-v1:0,Nova 2 Lite,"TEXT, IMAGE, VIDEO",TEXT,True,,INFERENCE_PROFILE
3,Amazon,amazon.nova-2-lite-v1:0:256k,Nova 2 Lite,"TEXT, IMAGE, VIDEO",TEXT,True,FINE_TUNING,PROVISIONED
4,Amazon,amazon.nova-2-multimodal-embeddings-v1:0,Amazon Nova Multimodal Embeddings,"TEXT, IMAGE, AUDIO, VIDEO",EMBEDDING,False,,ON_DEMAND
5,Amazon,amazon.nova-2-sonic-v1:0,Nova 2 Sonic,SPEECH,"SPEECH, TEXT",True,,ON_DEMAND
6,Amazon,amazon.nova-canvas-v1:0,Nova Canvas,"TEXT, IMAGE",IMAGE,False,FINE_TUNING,"ON_DEMAND, PROVISIONED"
7,Amazon,amazon.nova-lite-v1:0,Nova Lite,"TEXT, IMAGE, VIDEO",TEXT,True,,"ON_DEMAND, INFERENCE_PROFILE"
8,Amazon,amazon.nova-lite-v1:0:24k,Nova Lite,"TEXT, IMAGE, VIDEO",TEXT,True,,PROVISIONED
9,Amazon,amazon.nova-lite-v1:0:300k,Nova Lite,"TEXT, IMAGE, VIDEO",TEXT,True,"FINE_TUNING, DISTILLATION",PROVISIONED


## Step 2 — Secure endpoint usage and minimal smoke test

**Goal:** Confirm that SageMaker can call Bedrock safely.

**Secure usage principles:**
- do **not** hardcode keys in notebooks
- use the **SageMaker execution role**
- restrict model IDs through config or an allowlist
- start with a **small, cheap** smoke test
- fail early if the model ID is not approved

**Why this matters:**  
In enterprise settings, the first failure should happen **before** a costly or unsafe request is sent.

In [44]:
# Choose one classroom model that supports text generation through Converse.
# Available model for this class
"""
amazon.nova-lite-v1:0 
amazon.nova-micro-v1:0 
amazon.nova-pro-v1:0 
"""
BEDROCK_TEXT_MODEL = "amazon.nova-lite-v1:0"

print("Default text model:", BEDROCK_TEXT_MODEL)
print("Caller ARN:", sts.get_caller_identity()["Arn"])

Default text model: amazon.nova-lite-v1:0
Caller ARN: arn:aws:sts::502761807248:assumed-role/SageMakerExecutionRole-S3Controlled/SageMaker


In [45]:
# Call LLM and get a response
response = bedrock_runtime.converse(
    modelId=BEDROCK_TEXT_MODEL,
    system=[{"text": "You are a helpful enterprise AI assistant."}],
    messages=[
        {
        "role": "user",
        "content": [{"text": "In two lines, summarize the plot of the movie Inception."}]}
    ]   ,
    inferenceConfig={
    "maxTokens": 250,
    "temperature": 0.5
    }
)

print(response)

{'ResponseMetadata': {'RequestId': '26c1fa91-a639-48b8-8c13-383c20883fd2', 'HTTPStatusCode': 200, 'HTTPHeaders': {'date': 'Thu, 23 Apr 2026 16:57:45 GMT', 'content-type': 'application/json', 'content-length': '511', 'connection': 'keep-alive', 'x-amzn-requestid': '26c1fa91-a639-48b8-8c13-383c20883fd2'}, 'RetryAttempts': 0}, 'output': {'message': {'role': 'assistant', 'content': [{'text': "Inception follows a skilled thief, Dom Cobb, who steals secrets from within the subconscious during the dream state. He is offered a chance to have his criminal record erased if he can successfully plant an idea in someone's mind, leading to a complex, layered mission fraught with danger and moral ambiguity."}]}}, 'stopReason': 'end_turn', 'usage': {'inputTokens': 20, 'outputTokens': 60, 'totalTokens': 80}, 'metrics': {'latencyMs': 657}}


In [46]:
print(response["usage"])

{'inputTokens': 20, 'outputTokens': 60, 'totalTokens': 80}


In [47]:
print(response["ResponseMetadata"])

{'RequestId': '26c1fa91-a639-48b8-8c13-383c20883fd2', 'HTTPStatusCode': 200, 'HTTPHeaders': {'date': 'Thu, 23 Apr 2026 16:57:45 GMT', 'content-type': 'application/json', 'content-length': '511', 'connection': 'keep-alive', 'x-amzn-requestid': '26c1fa91-a639-48b8-8c13-383c20883fd2'}, 'RetryAttempts': 0}


In [48]:
print(response["output"])

{'message': {'role': 'assistant', 'content': [{'text': "Inception follows a skilled thief, Dom Cobb, who steals secrets from within the subconscious during the dream state. He is offered a chance to have his criminal record erased if he can successfully plant an idea in someone's mind, leading to a complex, layered mission fraught with danger and moral ambiguity."}]}}


In [49]:
print(response["output"]["message"]["content"][0]["text"])

Inception follows a skilled thief, Dom Cobb, who steals secrets from within the subconscious during the dream state. He is offered a chance to have his criminal record erased if he can successfully plant an idea in someone's mind, leading to a complex, layered mission fraught with danger and moral ambiguity.


In [50]:
# Now do the same with a helper function that we can reuse for the rest of the notebook
def ask_llm(user_prompt: str,
            system_prompt: str = "You are a helpful enterprise AI assistant.",
            model_id: str = BEDROCK_TEXT_MODEL,
            max_tokens: int = 250,
            temperature: float = 0.2):

    response = bedrock_runtime.converse(
        modelId=model_id,
        system=[{"text": system_prompt}],
        messages=[
            {
                "role": "user",
                "content": [{"text": user_prompt}]
            }
        ],
        inferenceConfig={
            "maxTokens": max_tokens,
            "temperature": temperature
        }
    )
    return response["output"]["message"]["content"][0]["text"]


In [51]:
# Test the helper function with a new prompt
print(ask_llm("In two lines, explain what an LLM API call is."))

An LLM API call is a request made to a large language model's interface to process or generate text based on provided input. It allows developers to integrate natural language understanding and generation capabilities into their applications.


In [52]:
# Change the parameters to see how the response changes
print(ask_llm("In two lines, explain what an LLM API call is.",max_tokens=100, temperature=0.5))

An LLM API call is a request made to a language model's application programming interface to generate text based on provided input. It enables developers to integrate natural language processing capabilities into their applications.


In [ ]:
# Change prompt, token length and temperature and check your answer
# Your code goes here

## Step 3 — Prompt styles and reasoning flows

We will test three common patterns:
1. simple instruction
2. role-based instruction
3. structured reasoning flow

**Explain to learners:**  
Better prompts are not just longer prompts.  
Better prompts make the model's job easier by clarifying:
- role
- task
- output shape
- constraints
- reasoning path

In [19]:
# Simple instruction prompt style
prompt = "Summarize why validation matters in enterprise AI systems."
print(ask_llm(prompt))

Validation is crucial in enterprise AI systems for several reasons:

1. **Accuracy and Reliability**: Validation ensures that AI models produce accurate and reliable results. This is essential for making informed business decisions based on the AI's output.

2. **Risk Management**: By validating AI systems, enterprises can identify and mitigate potential risks, such as biases, errors, and security vulnerabilities, which could lead to significant financial and reputational damage.

3. **Compliance**: Many industries are subject to regulatory requirements. Validating AI systems helps ensure compliance with laws and standards, avoiding legal penalties and fostering trust with stakeholders.

4. **Performance Optimization**: Validation processes help in fine-tuning AI models to improve their performance, efficiency, and effectiveness, leading to better outcomes and cost savings.

5. **User Trust and Adoption**: Validating AI systems builds confidence among users and stakeholders. When users

In [20]:
# Role based prompt style
prompt = """You are an enterprise AI expert. 
            Summarize why validation matters in enterprise AI systems."""
print(ask_llm(prompt))

Validation is a critical component of enterprise AI systems for several reasons:

1. **Accuracy and Reliability**: Validation ensures that the AI models produce accurate and reliable results. This is essential for making informed business decisions, as stakeholders rely on the AI's output for strategic planning and operational efficiency.

2. **Risk Management**: Validating AI systems helps identify and mitigate potential risks. This includes detecting biases, errors, and vulnerabilities that could lead to incorrect predictions or unethical outcomes.

3. **Compliance and Governance**: Many industries are subject to regulatory requirements. Validating AI systems ensures compliance with these regulations and helps maintain corporate governance standards.

4. **Trust and Transparency**: Stakeholders, including customers, partners, and employees, need to trust the AI systems. Validation processes provide transparency and build confidence in the AI's performance and decision-making processe

In [21]:
# Reasoning flow prompt style
prompt = """You are an enterprise AI expert. 
            Summarize why validation matters in enterprise AI systems.
            Explain your reasoning step by step."""
print(ask_llm(prompt))

Certainly! Validation is a critical component of enterprise AI systems for several reasons. Here’s a step-by-step explanation of why validation matters:

### 1. **Ensuring Accuracy and Reliability**
   - **Reasoning**: Enterprise AI systems are often used to make important decisions that can impact business operations, customer experiences, and financial outcomes. Ensuring the accuracy and reliability of these systems is paramount.
   - **Steps**:
     - **Training Data Quality**: Validate that the training data is accurate, representative, and free from biases.
     - **Model Performance**: Regularly test the model’s performance using validation datasets to ensure it meets predefined accuracy metrics.
     - **Continuous Monitoring**: Implement ongoing monitoring to detect and correct any deviations in performance over time.

### 2. **Mitigating Risks**
   - **Reasoning**: AI systems can introduce various risks, including financial losses, legal liabilities, and reputational damage. V

In [58]:
# Write one simple instruction prompt and one role based prompt and execute. 
# You can also play with temperature and max tokens

In [57]:
# Your code goes here

## System Prompts

In [22]:
# System message prompt style
prompt = "Summarize why validation matters in enterprise AI systems."
system_prompt = "You are an enterprise AI expert. Please provide a detailed answer."        
print(ask_llm(prompt, system_prompt=system_prompt))

Validation is a critical component of enterprise AI systems for several reasons:

### 1. **Ensuring Accuracy and Reliability**
Validation processes help ensure that AI models are accurate and reliable. In enterprise settings, decisions based on AI outputs can have significant consequences, such as financial transactions, hiring decisions, or healthcare diagnoses. Accurate and reliable AI systems are essential to maintain trust and integrity in these processes.

### 2. **Mitigating Bias and Fairness**
Validation helps identify and mitigate biases in AI models. Biased models can lead to unfair treatment of certain groups, which can have legal, ethical, and reputational consequences for enterprises. Comprehensive validation ensures that AI systems treat all users equitably.

### 3. **Compliance with Regulations**
Many industries are subject to regulatory requirements that mandate the use of validated AI systems. For example, the healthcare industry must comply with regulations like HIPAA,

In [23]:
# Another example of System message prompt style with a more specific system prompt
prompt = "Summarize why validation matters in enterprise AI systems."
system_prompt = """You are a seasoned enterprise AI consultant with 20 years of experience.
                     You have worked with Fortune 500 companies across various industries to help them implement and validate their AI systems effectively. 
                     Please provide a detailed answer on why validation is crucial in enterprise AI systems, drawing from your extensive experience."""     
print(ask_llm(prompt, system_prompt=system_prompt))

Validation is a cornerstone of successful enterprise AI systems, and its importance cannot be overstated. Here’s a detailed explanation of why validation is crucial, drawing from my extensive experience working with Fortune 500 companies across various industries:

### 1. **Ensuring Accuracy and Reliability**
Validation ensures that AI models perform accurately and reliably under real-world conditions. In enterprise settings, the stakes are high—decisions made by AI systems can impact financial outcomes, customer satisfaction, and operational efficiency. Inaccurate or unreliable AI can lead to costly errors, such as incorrect predictions, misclassifications, or faulty recommendations.

### 2. **Building Trust and Confidence**
For stakeholders, including executives, employees, and customers, trust is paramount. Validation processes demonstrate that the AI system has been rigorously tested and meets the necessary performance standards. This builds confidence in the technology, facilitati

In [24]:
# More complex prompt styles with few-shot examples can also be used
prompt = "You are an enterprise AI expert."
system_prompt = """You are an enterprise AI expert.
Here are some examples of how to answer questions about enterprise AI systems:                  
Q: Why is validation important in enterprise AI systems?
A: Validation is crucial in enterprise AI systems to ensure that the models are performing as expected,
    to identify any biases or issues before deployment, and to maintain trust with stakeholders by demonstrating that the AI system is reliable and effective.  
Q: What are some common techniques for validating enterprise AI systems?    
A: Common techniques for validating enterprise AI systems include cross-validation, A/B testing, monitoring performance metrics in production, and conducting regular audits to check for bias and drift."""
print(ask_llm(prompt, system_prompt=system_prompt))


Q: How can enterprises ensure the ethical use of AI in their systems?

A: Enterprises can ensure the ethical use of AI in their systems by implementing a comprehensive framework that includes several key practices. First, it's important to establish clear ethical guidelines and principles that align with the organization's values and regulatory requirements. This involves engaging stakeholders, including employees, customers, and ethical experts, to identify potential ethical issues and ensure diverse perspectives are considered.

Second, enterprises should conduct thorough risk assessments to identify and mitigate potential biases and unfair impacts on different user groups. This includes using diverse and representative datasets for training AI models and regularly auditing the models for bias and discrimination.

Third, transparency and explainability are critical. Enterprises should implement mechanisms to make AI decisions understandable to users and stakeholders, ensuring that th

In [25]:
# Change system prompt to agent tell the model it is an agent that needs to answer the question and take action if needed.
prompt = "Summarize why validation matters in enterprise AI systems."
system_prompt = """You are an enterprise AI agent. 
                Your task is to answer the user's question."""
print(ask_llm(prompt, system_prompt=system_prompt))

Validation is crucial in enterprise AI systems for several reasons:

1. **Accuracy and Reliability**: Validation ensures that AI models produce accurate and reliable results. This is essential for making informed business decisions based on AI insights.

2. **Risk Management**: Validating AI systems helps identify and mitigate potential risks, such as biases, errors, and security vulnerabilities, which can lead to costly mistakes.

3. **Compliance**: Many industries have regulatory requirements that mandate the validation of AI systems to ensure they meet legal standards and protect consumer rights.

4. **Trust and Transparency**: Validating AI systems builds trust among stakeholders, including customers, employees, and partners, by demonstrating that the AI is reliable and operates transparently.

5. **Performance Optimization**: Validation allows for continuous monitoring and improvement of AI models, ensuring they perform optimally over time as new data and conditions emerge.

6. **

# LLM Interaction Logging & Observability Framework

Build a reusable LLM interface that not only generates responses but also captures structured telemetry for monitoring, cost tracking, and analysis.

In [59]:
import os
import pandas as pd
from datetime import datetime

LOG_CSV_PATH = "llm_prompt_log.csv"

def ask_llm(
    user_prompt: str,
    asked_by: str,
    system_prompt: str = "You are a helpful enterprise AI assistant.",
    model_id: str = BEDROCK_TEXT_MODEL,
    max_tokens: int = 250,
    temperature: float = 0.2,
    csv_path: str = LOG_CSV_PATH
) -> str:
    """
    Call the Bedrock model, return the answer,
    and log prompt/response details into a CSV using pandas.
    """

    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    response = bedrock_runtime.converse(
        modelId=model_id,
        system=[{"text": system_prompt}],
        messages=[
            {
                "role": "user",
                "content": [{"text": user_prompt}]
            }
        ],
        inferenceConfig={
            "maxTokens": max_tokens,
            "temperature": temperature
        }
    )

    # Extract answer text
    answer = response["output"]["message"]["content"][0]["text"]

    # Extract usage safely
    usage = response.get("usage", {})
    input_tokens = usage.get("inputTokens", None)
    output_tokens = usage.get("outputTokens", None)
    total_tokens = usage.get("totalTokens", None)

    # Create one log row
    new_row = pd.DataFrame([{
        "timestamp": timestamp,
        "asked_by": asked_by,
        "model_id": model_id,
        "system_prompt": system_prompt,
        "user_prompt": user_prompt,
        "response_text": answer,
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "total_tokens": total_tokens,
        "max_tokens": max_tokens,
        "temperature": temperature
    }])

    # Append to existing CSV or create new one
    if os.path.exists(csv_path):
        existing_df = pd.read_csv(csv_path)
        updated_df = pd.concat([existing_df, new_row], ignore_index=True)
    else:
        updated_df = new_row

    updated_df.to_csv(csv_path, index=False)

    return answer

In [63]:
user_prompt = "Summarize why validation matters in enterprise AI systems."
system_prompt = """You are an enterprise AI agent. 
                Your task is to answer the user's question."""
user = "user@123"
ask_llm(user_prompt, system_prompt=system_prompt,asked_by=user)

'Validation is crucial in enterprise AI systems for several reasons:\n\n1. **Accuracy and Reliability**: Validation ensures that AI models produce accurate and reliable results. This is essential for decision-making processes that rely on AI insights.\n\n2. **Compliance and Governance**: Many industries are subject to regulatory requirements. Validating AI systems helps ensure compliance with these regulations and standards, avoiding legal and financial repercussions.\n\n3. **Risk Management**: Validating AI systems helps identify and mitigate potential risks, such as biases, errors, and security vulnerabilities, which could lead to significant operational disruptions or financial losses.\n\n4. **User Trust**: Reliable and validated AI systems build trust among users and stakeholders. Trust is critical for the adoption and effective use of AI technologies within an organization.\n\n5. **Performance Optimization**: Validation processes help in fine-tuning models to improve performance, 

## Reflection checkpoint
Discuss pros and cons of prompt style before move to next section

## Step 4 — Build a prompt-to-action pipeline

A production-safe workflow often looks like this:
1. understand the request
2. convert it into a structured plan
3. validate the plan
4. execute only the allowed action
5. call a controlled backend function
6. log the result

This is the core **tool/function calling pattern** we want learners to understand.

### Important note
This notebook uses a **model-agnostic structured JSON plan**.  
That keeps the concept simple and portable across models.

In [26]:
ACTION_PLANNER_PROMPT = '''
You are an action planner for an enterprise assistant.

Return valid JSON only.
Use this schema:
{
  "intent": "<one short label>",
  "needs_backend_action": true or false,
  "action_name": "<action or none>",
  "arguments": { ... },
  "reason_for_action": "<one sentence>"
}

Allowed action names:
- create_ticket
- lookup_user_record
- none

Examples:
User: "Create an IT ticket for VPN access issue"
Output:
{"intent":"create_it_ticket","needs_backend_action":true,"action_name":"create_ticket","arguments":{"category":"IT Support","summary":"VPN access issue"},"reason_for_action":"A ticket must be created in the backend system."}

User: "Find the team and location for ananya"
Output:
{"intent":"lookup_user_record","needs_backend_action":true,"action_name":"lookup_user_record","arguments":{"user_name":"ananya"},"reason_for_action":"The answer should come from the CSV file loaded from S3."}
'''


In [27]:
# Function to extract JSON object from LLM response
def extract_json_object(text: str) -> dict:
    return json.loads(text.strip())

# Function to plan action based on user request
def plan_action(user_request: str) -> dict:
    raw = ask_llm(
        user_prompt=f"{ACTION_PLANNER_PROMPT}\n\nUser request: {user_request}",
        system_prompt="You convert user requests into safe structured action plans.",
        max_tokens=300,
        temperature=0
    )
    return extract_json_object(raw)

In [28]:
# Test the action planner with a sample user request
plan_action("Create an IT ticket for laptop replacement")

{'intent': 'create_it_ticket',
 'needs_backend_action': True,
 'action_name': 'create_ticket',
 'arguments': {'category': 'IT Support', 'summary': 'laptop replacement'},
 'reason_for_action': 'A ticket must be created in the backend system.'}

In [30]:
# Test the action planner with another sample user request
plan_action("Find the team and location for ananya")

{'intent': 'lookup_user_record',
 'needs_backend_action': True,
 'action_name': 'lookup_user_record',
 'arguments': {'user_name': 'ananya'},
 'reason_for_action': 'The team and location information should be retrieved from the backend system.'}

In [37]:
import pandas as pd
# Read data from S3 to be used for backend action execution
S3_DATA_BUCKET = "s3-demo-bucket-adp"
S3_FILE_PATH = "training_user_directory.csv"

def read_csv_from_s3(bucket: str, file_path: str) -> pd.DataFrame:
    obj = s3.get_object(Bucket=bucket, Key=file_path)
    return pd.read_csv(obj["Body"])

df = read_csv_from_s3(S3_DATA_BUCKET, S3_FILE_PATH)
print(df.head())

  user_name    full_name                  team   location      manager  \
0    ananya   Ananya Rao      Data Engineering  Bengaluru  Rahul Mehta   
1    vikram  Vikram Shah  Platform Engineering  Hyderabad   Priya Nair   
2      neha   Neha Gupta         QA Automation       Pune  Sandeep Rao   
3     arjun   Arjun Iyer            IT Support    Chennai  Meera Joshi   
4      sara    Sara Khan               Product  Hyderabad    Dev Malik   

     laptop_type  
0    MacBook Air  
1   ThinkPad T14  
2    MacBook Pro  
3    ThinkPad X1  
4  Dell Latitude  


In [38]:
# Function to execute the planned action
def execute_plan(plan: dict) -> dict:
    action_name = plan["action_name"]

    if action_name == "lookup_user_record":
        user_name = plan["arguments"]["user_name"]
        df = read_csv_from_s3(S3_DATA_BUCKET, S3_FILE_PATH)

        match = df[df["user_name"].str.lower() == user_name.lower()]

        if match.empty:
            return {
                "status": "not_found",
                "message": f"No user found for '{user_name}'."
            }

        row = match.iloc[0]

        return {
            "status": "success",
            "message": "User found.",
            "user_record": {
                "user_name": row["user_name"],
                "full_name": row["full_name"],
                "team": row["team"],
                "location": row["location"],
                "manager": row["manager"],
                "laptop_type": row["laptop_type"]
            }
        }

    return {
        "status": "no_action",
        "message": "No backend action required."
    }


In [39]:
# run the full pipeline with a sample user request
def run_prompt_to_action_pipeline(user_request: str) -> dict:
    plan = plan_action(user_request)
    result = execute_plan(plan)
    return {
        "user_request": user_request,
        "plan": plan,
        "result": result
    }

In [40]:
# Test the full pipeline with a sample user request
response = run_prompt_to_action_pipeline("Find the team and location for ananya")
print(json.dumps(response, indent=2))

{
  "user_request": "Find the team and location for ananya",
  "plan": {
    "intent": "lookup_user_record",
    "needs_backend_action": true,
    "action_name": "lookup_user_record",
    "arguments": {
      "user_name": "ananya"
    },
    "reason_for_action": "The answer should come from the CSV file loaded from S3."
  },
  "result": {
    "status": "success",
    "message": "User found.",
    "user_record": {
      "user_name": "ananya",
      "full_name": "Ananya Rao",
      "team": "Data Engineering",
      "location": "Bengaluru",
      "manager": "Rahul Mehta",
      "laptop_type": "MacBook Air"
    }
  }
}


In [41]:
# Test the full pipeline with a sample user request
response = run_prompt_to_action_pipeline("Find the team and location for karthik")
print(json.dumps(response, indent=2))

{
  "user_request": "Find the team and location for karthik",
  "plan": {
    "intent": "lookup_user_record",
    "needs_backend_action": true,
    "action_name": "lookup_user_record",
    "arguments": {
      "user_name": "karthik"
    },
    "reason_for_action": "The answer should come from the CSV file loaded from S3."
  },
  "result": {
    "status": "not_found",
    "message": "No user found for 'karthik'."
  }
}
